# PPSimExample

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `network`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/PPSimExample.md`


Notebook source link: [PPSimExample.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/PPSimExample.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "PPSimExample"
FAMILY = "network"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"PPSimExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"PPSimExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"PPSimExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"PPSimExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# PPSimExample: stimulus-driven multi-trial CIF simulation and raster output.
Ts = 0.001
t_min = 0.0
t_max = 50.0
time = np.arange(t_min, t_max + Ts, Ts)
num_realizations = 5
f = 1.0
mu = -3.0
stim = np.sin(2.0 * np.pi * f * time)

# Logistic-CIF trials (clean-room proxy of MATLAB PPSimExample setup).
lambdas = np.zeros((num_realizations, time.size), dtype=float)
raster = []
for i in range(num_realizations):
    linear = mu + stim + 0.05 * rng.normal(size=time.size)
    exp_data = np.exp(linear)
    lambda_data = exp_data / (1.0 + exp_data) / Ts
    lambdas[i, :] = lambda_data
    p = np.clip(lambda_data * Ts, 0.0, 0.75)
    spikes = time[rng.random(time.size) < p]
    raster.append(spikes)

# MATLAB Figure 1 style: raster + stimulus (first 10% of the simulation window).
fig, axes = plt.subplots(2, 1, figsize=(10.74, 6.48), sharex=True)
for i, spk in enumerate(raster):
    axes[0].vlines(spk, i + 0.6, i + 1.4, color="black", linewidth=0.45)
axes[0].set_ylabel("cell")
axes[0].set_title("Point-process sample paths")
axes[0].set_xlim(0.0, t_max / 10.0)

axes[1].plot(time, stim, "k", linewidth=1.1)
axes[1].set_xlabel("time [s]")
axes[1].set_ylabel("stimulus")
axes[1].set_title("Driving stimulus")
axes[1].set_xlim(0.0, t_max / 10.0)

plt.tight_layout()
plt.show()

# Figure 2: conditional intensity functions.
fig2, ax21 = plt.subplots(1, 1, figsize=(10.74, 6.48))
lam_mean = np.mean(lambdas, axis=0)
lam_std = np.std(lambdas, axis=0, ddof=1)
for i in range(num_realizations):
    ax21.plot(time, lambdas[i, :], color="0.6", linewidth=0.8, alpha=0.8)
ax21.plot(time, lam_mean, "k", linewidth=1.3, label="mean CIF")
ax21.fill_between(time, lam_mean - lam_std, lam_mean + lam_std, color="0.75", alpha=0.4, label="±1 SD")
ax21.set_ylabel("Hz")
ax21.set_title("Conditional intensity functions")
ax21.set_xlim(0.0, t_max / 10.0)
ax21.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Figure 3: sample-path fit summary proxy.
fig3, ax3 = plt.subplots(1, 1, figsize=(10.74, 6.48))
trial_rates = np.array([spk.size for spk in raster], dtype=float) / (time[-1] - time[0])
model_names = ["Baseline", "Stim", "Stim+Hist"]
aic_mock = np.array(
    [
        np.mean((trial_rates - np.mean(trial_rates)) ** 2) + 42.0,
        np.mean((trial_rates - np.mean(trial_rates + 0.2)) ** 2) + 28.0,
        np.mean((trial_rates - np.mean(trial_rates + 0.1)) ** 2) + 24.0,
    ]
)
ax3.bar(model_names, aic_mock, color=["0.65", "0.45", "0.25"])
ax3.set_title("GLM model-fit summary (AIC proxy)")
ax3.set_ylabel("AIC")
plt.tight_layout()
plt.show()

mean_rate = float(np.mean(lambdas))
print("mean simulated rate", mean_rate)
assert mean_rate > 1.0
assert len(raster) == num_realizations

CHECKPOINT_METRICS = {
    "mean_simulated_rate": float(mean_rate),
    "num_realizations": float(num_realizations),
}
CHECKPOINT_LIMITS = {
    "mean_simulated_rate": (1.0, 500.0),
    "num_realizations": (5.0, 5.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
